# SOUL AI — G3.1 Colab evaluation

Notebook chạy inference/evaluation cho cả hai frozen world-model checkpoint G3.1 (A và B1) trên cùng dữ liệu Pendulum PILOT.

- Checkpoint A và B1: chỉ TRAIN (seeds 1001–1005); B1 là bản 300-update.
- Đánh giá: PILOT đã ghi nhận (seeds 2001–2005).
- Baselines: persistence và mean-delta chỉ fit từ TRAIN.
- Dùng PyTorch/NumPy có sẵn trong Colab; không cài lại bộ CUDA nặng từ lockfile.
- Không huấn luyện lại, không ghi đè artifact trong repo, không mở CONFIRM.
- Bộ chấm bỏ qua transition đầu mỗi episode làm warm-up theo đúng cách trainer khởi tạo RSSM; 95/100 transition được chấm (19 × 5 episode).
- Kết quả luôn được gắn nhãn exploratory; threshold chưa khóa nên không có kết luận PASS/FAIL về chất lượng mô hình.

> Đây là bài kiểm tra world-model dự đoán trạng thái tiếp theo trong Pendulum-v1, không phải chatbot và cũng chưa phải agent tự chơi Minecraft.

In [ ]:
# 1) Clone đúng nhánh Colab, không dùng main thay thế
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/nguyenphuck2468-tech/Soul-AI.git"
BRANCH = "colab/g31-pilot-evaluation"
REPO_DIR = pathlib.Path("/content/Soul-AI")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Colab already ships with PyTorch and NumPy.
# Reinstalling the full lockfile would download large CUDA wheels unnecessarily.
import numpy as np
import torch

torch.set_num_threads(2)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Scoring device: CPU (deterministic evaluation; GPU is not required)")

In [ ]:
# 3) Verify protocol boundaries, both checkpoint provenances, and frozen hashes
import hashlib, json, torch
from pathlib import Path

subprocess.run([sys.executable, "-m", "scripts.phase0_validate_protocol"], check=True)

checkpoints = {
    "A": Path("artifacts/g31/g31_verified_train_checkpoint.pt"),
    "B1": Path("artifacts/g31/g31_b1_checkpoint.pt"),
}
expected_hashes = {
    "A": "0fd3fbc46eadd3e6dc4b2d9af873f62ef4c5a04477bfedd06b8565d704b32147",
    "B1": "136017b650fc910043c559822d47fea900391e2823324413327de815a4d8c018",
}
pilot_path = Path("artifacts/g31/pilot.jsonl")
train_path = Path("artifacts/g31/train.jsonl")

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path in [*checkpoints.values(), pilot_path, train_path]:
    if not path.is_file():
        raise FileNotFoundError(path)

checkpoint_metadata = {}
for label, path in checkpoints.items():
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    digest = sha256(path)
    assert digest == expected_hashes[label], (label, digest)
    assert checkpoint.get("split") == "TRAIN", (label, checkpoint.get("split"))
    assert checkpoint.get("data_split") == "TRAIN", (label, checkpoint.get("data_split"))
    assert sorted(checkpoint.get("train_seeds", [])) == [1001, 1002, 1003, 1004, 1005]
    assert checkpoint.get("environment_id") == "Pendulum-v1"
    checkpoint_metadata[label] = {
        "path": str(path),
        "sha256": digest,
        "training_updates": checkpoint.get("training_updates"),
        "train_seeds": checkpoint.get("train_seeds"),
    }
    print(label, json.dumps(checkpoint_metadata[label], indent=2))

print("TRAIN dataset SHA-256:", sha256(train_path))
print("PILOT dataset SHA-256:", sha256(pilot_path))
print("CONFIRM: not loaded")

In [ ]:
# 4) Score both frozen checkpoints with the official evaluator (zero optimizer updates)
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path("work") / "colab_g31_eval" / run_id
RUN_ROOT.mkdir(parents=True, exist_ok=False)
RUNS = {}

runtime_info = {
    "repo_branch": subprocess.check_output(
        ["git", "branch", "--show-current"], text=True
    ).strip(),
    "repo_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "python": sys.version,
    "torch": torch.__version__,
    "numpy": np.__version__,
    "device": "cpu",
    "cuda_available_but_unused": bool(torch.cuda.is_available()),
    "checkpoint_hashes": expected_hashes,
}
(RUN_ROOT / "runtime.json").write_text(
    json.dumps(runtime_info, indent=2), encoding="utf-8"
)
print("Runtime:", json.dumps(runtime_info, indent=2))

for label, checkpoint_path in checkpoints.items():
    out = RUN_ROOT / label
    out.mkdir(parents=True, exist_ok=False)
    cmd = [
        sys.executable, "-m", "soul.g31.score_pilot",
        "--checkpoint", str(checkpoint_path),
        "--predictions", str(out / "predictions.jsonl"),
        "--metrics", str(out / "metrics.json"),
        "--report", str(out / "report.json"),
    ]
    print(f"Running {label}:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    RUNS[label] = out

print("Output root:", RUN_ROOT)

In [ ]:
# 5) Inspect both results and enforce the no-CONFIRM/no-training contract
import json

RESULTS = {}
REPORTS = {}
for label, out in RUNS.items():
    metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
    report = json.loads((out / "report.json").read_text(encoding="utf-8"))
    assert report["confirm_accessed"] is False
    assert report["optimizer_updates"] == 0
    assert report["threshold"] == "TO_LOCK"
    assert report["gate"] == "INCONCLUSIVE"
    assert report["transitions"] == 95
    assert report["recorded_transitions"] == 100
    assert report["warmup_transitions_skipped"] == 5
    assert metrics["transition_count"] == 95
    assert metrics["threshold"] == "TO_LOCK"
    assert metrics["pilot_scoring"].startswith("EXPLORATORY")
    RESULTS[label] = metrics
    REPORTS[label] = report

    print(f"\\n{label} report:")
    print(json.dumps(report, indent=2, ensure_ascii=False))

def show_aggregate(label, payload):
    print("\\n" + label)
    for metric, values in payload.items():
        print(
            f"{metric:16s} mean={values['mean']:.6f} "
            f"median={values['median']:.6f} "
            f"dispersion={values['dispersion']:.6f} "
            f"episodes={values['episode_count']}"
        )

for label, metrics in RESULTS.items():
    show_aggregate(f"{label} SOUL G3.1 model", metrics["model_aggregate_equal_episode"])

# Both checkpoints use the same TRAIN-only baselines and the same PILOT episodes.
show_aggregate("Persistence baseline", RESULTS["A"]["persistence_aggregate_equal_episode"])
show_aggregate("TRAIN-only mean-delta baseline", RESULTS["A"]["mean_delta_aggregate_equal_episode"])

print("\\nPrimary metric comparison (theta RMSE; lower is better):")
for label in ("A", "B1"):
    print(label, RESULTS[label]["model_aggregate_equal_episode"]["theta_rmse"]["mean"])
print("B1 - A:", (
    RESULTS["B1"]["model_aggregate_equal_episode"]["theta_rmse"]["mean"]
    - RESULTS["A"]["model_aggregate_equal_episode"]["theta_rmse"]["mean"]
))

print("\\nSupplemental velocity RMSE:")
for label in ("A", "B1"):
    print(label, RESULTS[label]["model_aggregate_equal_episode"]["velocity_rmse"]["mean"])

print("\\nInterpretation: descriptive PILOT metrics only; threshold remains TO_LOCK.")

In [ ]:
# 6) Plot actual vs predicted for A and B1 on the same PILOT episode
import matplotlib.pyplot as plt
import numpy as np

records_by_label = {
    label: [
        json.loads(line)
        for line in (out / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
    ]
    for label, out in RUNS.items()
}
episode_ids = sorted({row["episode_id"] for row in records_by_label["A"]})
selected_episode = episode_ids[0]

def episode_rows(label):
    return sorted(
        [row for row in records_by_label[label] if row["episode_id"] == selected_episode],
        key=lambda row: row["timestep"],
    )

rows_a = episode_rows("A")
steps = [row["timestep"] for row in rows_a]
target = np.asarray([row["target_physical"] for row in rows_a])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(steps, target[:, 0], label="Actual theta")
axes[1].plot(steps, target[:, 1], label="Actual angular velocity")
for label in ("A", "B1"):
    rows = episode_rows(label)
    assert [row["timestep"] for row in rows] == steps
    pred = np.asarray([row["predicted_physical"] for row in rows])
    axes[0].plot(steps, pred[:, 0], label=f"{label} predicted theta")
    axes[1].plot(steps, pred[:, 1], label=f"{label} predicted velocity")

axes[0].set_ylabel("Angle (rad)")
axes[1].set_ylabel("Angular velocity (rad/s)")
axes[1].set_xlabel("Timestep")
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.25)
fig.suptitle(f"G3.1 PILOT episode: {selected_episode}")
plt.tight_layout()
plt.show()

In [ ]:
# 7) Package both checkpoints' predictions, metrics, and reports for download
import shutil
from google.colab import files

archive_base = f"/content/SOUL_AI_G31_Colab_{run_id}"
archive_path = shutil.make_archive(archive_base, "zip", root_dir=RUN_ROOT)
print("Created:", archive_path)
files.download(archive_path)

## Cách đọc kết quả

- **theta** được so sánh theo sai số góc tuần hoàn; **velocity** theo rad/s.
- Mỗi episode được tính riêng rồi tổng hợp ngang trọng số episode, tránh episode dài lấn át.
- So sánh model với persistence và mean-delta baseline; xem cả trung bình, trung vị và độ phân tán.
- Nếu model kém baseline, đó là kết quả âm cần giữ lại, không được che đi bằng cách đổi seed hoặc sửa threshold sau khi xem kết quả.
- Notebook chỉ đánh giá PILOT đã tồn tại trong repo. Không chạy lệnh thu thập, huấn luyện, khóa threshold, hay bất kỳ lệnh CONFIRM nào.

Tham khảo mã đánh giá gốc: soul/g31/score_pilot.py và soul/g31/evaluator.py.